# Lesson 34: Graph Data & Network Analysis

*ML Academy — companion notebook*

Adjacency matrices, centrality and PageRank, community detection and modularity, link prediction, and graph features for ordinary models.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-34.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Representing a graph

In [1]:
import numpy as np
rng = np.random.default_rng(2)
sizes = [40, 35, 25]                                            # three hidden communities
group = np.repeat([0, 1, 2], sizes); n = len(group)
p_in, p_out = 0.20, 0.01
P = np.where(group[:, None] == group[None, :], p_in, p_out)
A = np.triu(rng.random((n, n)) < P, 1).astype(int); A = A + A.T   # undirected, no self-loops
deg = A.sum(axis=1)
print("nodes", n, " edges", A.sum() // 2, " average degree", round(deg.mean(), 2))
print("edges inside communities:", int((A * (group[:, None] == group[None, :])).sum() // 2), "  between:", int((A * (group[:, None] != group[None, :])).sum() // 2))

nodes 100  edges 357  average degree 7.14
edges inside communities: 332   between: 25


## 3. Who matters? Centrality

In [2]:
def pagerank(A, d=0.85, iters=100):
    n = len(A); out = A.sum(axis=1)
    M = np.where(out[:, None] > 0, A / np.maximum(out[:, None], 1), 1 / n)   # row-stochastic transitions
    r = np.full(n, 1 / n)
    for _ in range(iters):
        r = (1 - d) / n + d * r @ M
    return r
def closeness(A):
    n = len(A); c = np.zeros(n)
    for s in range(n):                                          # breadth-first search from each node
        dist = np.full(n, -1); dist[s] = 0; frontier, d = [s], 0
        while frontier:
            d += 1
            nxt = {v for u in frontier for v in np.nonzero(A[u])[0] if dist[v] < 0}
            for v in nxt: dist[v] = d
            frontier = list(nxt)
        reach = dist[dist > 0]; c[s] = len(reach) / reach.sum() if len(reach) else 0
    return c
pr, cl = pagerank(A), closeness(A)
for name, score in [("degree", deg), ("PageRank", pr), ("closeness", cl)]:
    top = np.argsort(-score)[:3]
    print(f"{name:9s} top nodes {top.tolist()}  (communities {group[top].tolist()})")
print("correlation of PageRank with degree:", round(np.corrcoef(pr, deg)[0, 1], 3))

degree    top nodes [41, 29, 78]  (communities [1, 0, 2])
PageRank  top nodes [41, 78, 29]  (communities [1, 2, 0])
closeness top nodes [15, 41, 29]  (communities [0, 1, 0])
correlation of PageRank with degree: 0.986


## 5. Finding communities

In [3]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score
def modularity(A, labels):
    m, k = A.sum() / 2, A.sum(axis=1)
    same = labels[:, None] == labels[None, :]
    return float(((A - np.outer(k, k) / (2 * m)) * same).sum() / (2 * m))
found = SpectralClustering(3, affinity="precomputed", random_state=0).fit_predict(A)
print("adjusted Rand index against the true communities:", round(adjusted_rand_score(group, found), 3))
print("modularity: found", round(modularity(A, found), 3), "  true", round(modularity(A, group), 3), "  random labels", round(modularity(A, rng.integers(0, 3, n)), 3))

adjusted Rand index against the true communities: 0.971
modularity: found 0.573   true 0.574   random labels -0.01


## 6. Predicting missing links

In [4]:
from sklearn.metrics import roc_auc_score
edges = np.argwhere(np.triu(A, 1)); rng.shuffle(edges)
hidden = edges[: len(edges) // 10]                               # hide 10% of the edges
A_obs = A.copy(); A_obs[hidden[:, 0], hidden[:, 1]] = A_obs[hidden[:, 1], hidden[:, 0]] = 0
non = np.argwhere(np.triu(1 - A, 1)); non = non[rng.choice(len(non), len(hidden), replace=False)]
pairs, truth = np.vstack([hidden, non]), np.r_[np.ones(len(hidden)), np.zeros(len(non))]
k_obs = A_obs.sum(axis=1)
common = np.array([(A_obs[u] & A_obs[v]).sum() for u, v in pairs])
adamic = np.array([sum(1 / np.log(k_obs[w]) for w in np.nonzero(A_obs[u] & A_obs[v])[0] if k_obs[w] > 1) for u, v in pairs])
pref = np.array([k_obs[u] * k_obs[v] for u, v in pairs])
for name, s in [("common neighbours", common), ("Adamic-Adar", adamic), ("preferential attachment", pref)]:
    print(f"{name:24s} AUC {roc_auc_score(truth, s):.3f}")

common neighbours        AUC 0.613
Adamic-Adar              AUC 0.605
preferential attachment  AUC 0.384


## 7. Graph features for ordinary models

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Each person has a noisy personal signal; predict whether they're in community 0.
target = (group == 0).astype(int)
signal = target + rng.normal(0, 1.2, n)
neigh_mean = (A @ signal) / np.maximum(deg, 1)                    # average signal of their contacts
for name, feats in [("own signal only", np.c_[signal]), ("plus contacts' average", np.c_[signal, neigh_mean])]:
    acc = cross_val_score(LogisticRegression(), feats, target, cv=5).mean()
    print(f"{name:23s} accuracy {acc:.3f}")

own signal only         accuracy 0.630
plus contacts' average  accuracy 0.720


## 9. In Python: a graph from an edge list

In [6]:
import numpy as np
import pandas as pd

pay = pd.DataFrame({"from": ["A", "A", "B", "C", "D", "E", "F", "F", "G", "H"],
                    "to":   ["B", "C", "C", "A", "E", "F", "D", "G", "H", "G"]})
nodes = sorted(set(pay["from"]) | set(pay["to"])); idx = {v: i for i, v in enumerate(nodes)}
A = np.zeros((len(nodes), len(nodes)))
for a, b in zip(pay["from"], pay["to"]):
    A[idx[a], idx[b]] = A[idx[b], idx[a]] = 1                  # treat payments as undirected links
comp, c = -np.ones(len(nodes), int), 0
for s in range(len(nodes)):                                    # connected components by search
    if comp[s] < 0:
        stack = [s]
        while stack:
            u = stack.pop()
            if comp[u] < 0:
                comp[u] = c; stack += list(np.nonzero(A[u])[0])
        c += 1
M, r = A / A.sum(axis=1, keepdims=True), np.full(len(nodes), 1 / len(nodes))
for _ in range(100):
    r = 0.15 / len(nodes) + 0.85 * r @ M
print(pd.DataFrame({"account": nodes, "degree": A.sum(axis=1).astype(int), "pagerank": r.round(3), "component": comp}).to_string(index=False))

account  degree  pagerank  component
      A       2     0.125          0
      B       2     0.125          0
      C       2     0.125          0
      D       2     0.120          1
      E       2     0.120          1
      F       3     0.177          1
      G       2     0.133          1
      H       1     0.075          1


## 12. Going deeper: PageRank as an eigenvector, and the graph Laplacian

In [7]:
import numpy as np
rng = np.random.default_rng(3)
n = 60
A = (rng.random((n, n)) < 0.08).astype(float); np.fill_diagonal(A, 0)      # a directed graph
out = A.sum(axis=1, keepdims=True)
M = np.where(out > 0, A / np.maximum(out, 1), 1 / n)
G = 0.15 / n + 0.85 * M                                                     # the "Google matrix"
vals, vecs = np.linalg.eig(G.T)
lead = np.real(vecs[:, np.argmax(np.real(vals))]); lead = lead / lead.sum()
r = np.full(n, 1 / n)
for it in range(1, 101):
    r_new = r @ G
    if np.abs(r_new - r).sum() < 1e-10: break
    r = r_new
print("largest eigenvalue:", round(float(np.max(np.real(vals))), 6), "  power iteration converged in", it, "steps")
print("max difference from the eigenvector:", f"{np.abs(r - lead).max():.1e}")
print("correlation with in-degree:", round(np.corrcoef(r, A.sum(axis=0))[0, 1], 3), "  with out-degree:", round(np.corrcoef(r, A.sum(axis=1))[0, 1], 3))

largest eigenvalue: 1.0   power iteration converged in 27 steps
max difference from the eigenvector: 3.6e-12
correlation with in-degree: 0.724   with out-degree: -0.208


In [8]:
sizes = [30, 30]; grp = np.repeat([0, 1], sizes)
P = np.where(grp[:, None] == grp[None, :], 0.25, 0.03)
B = np.triu(rng.random((60, 60)) < P, 1).astype(float); B = B + B.T
L = np.diag(B.sum(axis=1)) - B                                              # graph Laplacian
w, v = np.linalg.eigh(L)
fiedler = v[:, 1]                                                           # second-smallest eigenvector
split = (fiedler > 0).astype(int)
acc = max((split == grp).mean(), (split != grp).mean())
print("smallest Laplacian eigenvalues:", np.round(w[:3], 3).tolist())
print("splitting by the sign of the second eigenvector recovers the groups for", f"{acc:.0%}", "of nodes")

smallest Laplacian eigenvalues: [-0.0, 2.091, 3.272]
splitting by the sign of the second eigenvector recovers the groups for 100% of nodes


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
Z = StandardScaler().fit_transform(df[["size_m2", "dist_km", "age_yrs", "price_lakh"]])
D = euclidean_distances(Z); np.fill_diagonal(D, np.inf)
A = np.zeros_like(D)
for i, row in enumerate(D):                          # link each home to its 2 most similar homes
    for j in np.argsort(row)[:2]: A[i, j] = A[j, i] = 1
same_area = (df.area.to_numpy()[:, None] == df.area.to_numpy()[None, :])
print("similarity links:", int(A.sum() // 2), " of which within the same area:", int((A * same_area).sum() // 2))
print("most connected homes (id):", df.id[np.argsort(-A.sum(axis=1))[:3]].tolist())

similarity links: 18  of which within the same area: 6
most connected homes (id): [7, 2, 4]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Degrees from an adjacency matrix

Compute each node's degree.

In [11]:
import numpy as np
A = np.array([[0, 1, 1], [1, 0, 0], [1, 0, 0]])
deg = A.sum()  # ← sum per row, not overall
result = deg.tolist()
check(result, [2, 1, 1], hint='Sum each row.')

❌ Not yet. Hint: Sum each row.


### Challenge 2: Common neighbours

Count the neighbours that nodes 1 and 2 share.

In [12]:
import numpy as np
A = np.array([[0, 1, 1, 1], [1, 0, 1, 1], [1, 1, 0, 1], [1, 1, 1, 0]])
common = int(A[1].sum())  # ← count only shared neighbours
check(common, 2, hint='Combine the two rows with & and count.')

❌ Not yet. Hint: Combine the two rows with & and count.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Degrees from an adjacency matrix** — the missing line:

```python
deg = A.sum(axis=1)
```

**2. Common neighbours** — the missing line:

```python
common = int((A[1] & A[2]).sum())
```